In [ ]:
import numpy as np
import mne
import math

In [ ]:
S001R01 = ("S001R01.edf")
R01raw = mne.io.read_raw_edf(S001R01)

S001R02 = ("S001R02.edf")
R02raw = mne.io.read_raw_edf(S001R02)

In [ ]:
R01raw.load_data()
R02raw.load_data()

R01raw = R01raw.filter(1, 40)
R02raw = R02raw.filter(1, 40)

In [ ]:
R01timing = R01raw.n_times // R01raw.info['sfreq']
R02timing = R02raw.n_times // R02raw.info['sfreq']

In [ ]:
R01Dict = {}
R01duration = math.floor(R01timing / 4)

for i in range (R01duration):
    R01Copy = R01raw.copy()
    R01Start = i * 4
    R01End = R01Start + 4
    
    R01Dict.update({f"R01: {i}" :R01Copy.crop(R01Start, R01End)})

In [ ]:
R02Dict = {}
R02duration = math.floor(R02timing / 4)

for i in range (R02duration):
    R02Copy = R02raw.copy()
    R02Start = i * 4
    R02End = R02Start + 4
    
    R02Dict.update({f"R02: {i}" :R02Copy.crop(R02Start, R02End)})

In [ ]:
for i in range(len(R01Dict)):
    R01Dict[f"R01: {i}"].compute_psd(fmax=50).plot(picks="data", exclude="bads", amplitude=False)
    R01Dict[f"R01: {i}"].plot(duration=5, n_channels=30)

In [ ]:
for i in range(len(R02Dict)):
    R02Dict[f"R02: {i}"].compute_psd(fmax=50).plot(picks="data", exclude="bads", amplitude=False)
    R02Dict[f"R02: {i}"].plot(duration=5, n_channels=30)

In [ ]:
combindDict = {}

for  key, value in R01Dict.items():
    combindDict[key] = (value, "eyes open")

for  key, value in R02Dict.items():
    combindDict[key] = (value, "eyes closed")  
    

In [ ]:
def getBandPower(chunk, lowFreq, highFreq):
    chunkPsd = chunk.compute_psd()
    psds, freqs = chunkPsd.get_data(return_freqs=True)
    freqs = np.array(freqs)
    bandmask = (freqs >= lowFreq) & (freqs <= highFreq)
    print(freqs[bandmask])
    return (psds[:, bandmask].mean(axis = 1)).mean()

In [ ]:
features = []
labels = []
for key, value in combindDict.items():
    features.append(getBandPower(value[0], 8, 13))
    labels.append(value[1])

In [ ]:
features = np.array(features)
labels = np.array(labels)
features = features.reshape(-1,1)